In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import os

# =============================================================================
# WHAT CHANGED FROM v5 AND WHY
# =============================================================================
# v5 problem: t_end=300-500 → system always reaches steady state → pathogen
# always driven to near-zero → community_suppression ALWAYS = 1.0
# This makes the community-level label useless for GAT training.
#
# CHANGE 1: Non-steady-state measurement time
#   We now stop at T_MEASURE = 20.0 time units (roughly equivalent to ~24-48h
#   in real experiments, depending on dilution rate delta ≈ 0.05-0.20/h).
#   At this early time point, competition is underway but not complete, giving
#   suppression values spread across [0, 1] rather than all = 1.
#
# CHANGE 2: Smaller, controllable community sizes matching the paper
#   The paper uses communities of 2-50 strains (sampled from 100 total).
#   v5 used 80-180 members per broth — way too large, drives suppression to 1.
#   We now explicitly generate communities of sizes: 2, 4, 6, 8, 12, 16, 24, 48
#   matching the real experimental data structure. This means we generate
#   multiple communities per broth (one per size), giving a dataset structure
#   where each row is (strain, community) not (strain, broth).
#
# CHANGE 3: Dataset structure now mirrors the paper's community_dataset_noGO.csv
#   Output: one row per (strain, community_id)
#   Columns: strain features (ss_lum, genes, uptake, R0) + community metadata
#   (community_id, community_size, community_suppression)
#   This is exactly what GAT-SS and GAT-GO expect.
#
# CHANGE 4: ss_luminescence is now computed as a proper non-SS pairwise value
#   Each strain's individual suppression is measured at T_MEASURE, not SS.
#   This mirrors how the real data measures it (fixed time point on plate).
#
# CHANGE 5: Reproducible community sampling
#   We generate N_COMMUNITIES_PER_BROTH communities per broth by sampling
#   from the broth's species pool. For each community size we sample
#   N_SAMPLES_PER_SIZE times. This gives us ~1500-2000 communities total.
# =============================================================================

# ── Reproducibility ──────────────────────────────────────────────────────────
rng = np.random.default_rng(seed=42)

# ── Global constants ──────────────────────────────────────────────────────────
N_SPECIES    = 101          # 1 pathogen + 100 community strains (matches paper)
N_RESOURCES  = 20           # 20 nutrients (matches paper)
N_GENES      = 8            # 8 binary genes (matches paper)
N_BROTHS     = 10           # number of distinct nutrient environments

PATHOGEN_IDX  = 0
N_COMMUNITY   = N_SPECIES - 1   # 100 strains available (matches paper)
N_SPECIALIST  = N_COMMUNITY // 2
N_GENERALIST  = N_COMMUNITY - N_SPECIALIST

# ── KEY PARAMETER: measurement time (NOT steady state) ──────────────────────
# At delta=0.1/h, T_MEASURE=20 corresponds to roughly 2 dilution times.
# This is early enough that pathogen is not fully excluded in most communities.
# Tune this if suppression distribution is too bimodal.
T_MEASURE = 20.0   # time units — the main knob for suppression spread

# ── Community sizes to generate — matching real experimental data ─────────────
COMMUNITY_SIZES  = [2, 4, 6, 8, 12, 16, 24, 48]
N_SAMPLES_PER_SIZE = 15   # communities per size per broth
# Total communities ≈ 10 broths × 8 sizes × 15 samples = 1200

OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData"

# Sparsity parameters — same as v5
UPTAKE_SPARSITY = {
    "pathogen"   : (0.30, 0.55),
    "specialist" : (0.10, 0.25),
    "generalist" : (0.40, 0.65),
}
CROSSFEED_OUT_MIN = 2
CROSSFEED_OUT_MAX = 6

RESOURCE_NAMES = [f"c{k+1}"           for k in range(N_RESOURCES)]
SPECIES_NAMES  = [f"Species_{i+1:04d}" for i in range(N_SPECIES)]
BROTH_NAMES    = [f"Broth_{b+1:02d}"   for b in range(N_BROTHS)]

TYPE_COLORS = {
    "pathogen"   : "#E53935",
    "specialist" : "#1E88E5",
    "generalist" : "#43A047",
}

In [2]:
# =============================================================================
# 1.  SPECIES TRAITS
# =============================================================================
def build_species_traits():
    """
    Binary genomes → sigmoid uptake rates, masked to sparse support.
    Now uses N_GENES=8 and N_RESOURCES=20 to match the paper's dataset.
    """
    genes = rng.integers(0, 2, size=(N_SPECIES, N_GENES))
    W     = rng.normal(0.0, 0.5, size=(N_GENES, N_RESOURCES))

    c_matrix     = np.zeros((N_SPECIES, N_RESOURCES))
    uptake_mask  = np.zeros((N_SPECIES, N_RESOURCES), dtype=bool)
    species_type = []
    preferred_nutrient = []

    for i in range(N_SPECIES):
        if i == PATHOGEN_IDX:
            stype = "pathogen"
            pref  = -1
        elif 1 <= i <= N_SPECIALIST:
            stype = "specialist"
            pref  = (i - 1) % N_RESOURCES
        else:
            stype = "generalist"
            pref  = -1

        species_type.append(stype)
        preferred_nutrient.append(pref)

        lo, hi = UPTAKE_SPARSITY[stype]
        density  = rng.uniform(lo, hi)
        n_active = max(1, int(round(density * N_RESOURCES)))

        active_indices = rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()
        if pref >= 0 and pref not in active_indices:
            active_indices[0] = pref
        uptake_mask[i, active_indices] = True

        if stype == "specialist":
            bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES)
            bias[pref] += rng.uniform(2.5, 4.0)
        else:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)

        raw    = genes[i].astype(float) @ W + bias
        c_full = 1.0 / (1.0 + np.exp(-raw))
        c_matrix[i] = c_full * uptake_mask[i]

    return genes, c_matrix, uptake_mask, species_type, preferred_nutrient

In [3]:
# =============================================================================
# 2.  BROTH ENVIRONMENTS
# =============================================================================
def build_broths():
    """
    N_BROTHS distinct nutrient environments.
    Each broth defines R0 (initial nutrient concentrations), delta, w, l.
    The broth does NOT define the community membership — that is done
    separately when we sample communities of each size.
    """
    broths = []
    for b in range(N_BROTHS):
        dominant     = int(rng.integers(0, N_RESOURCES))
        non_dominant = [k for k in range(N_RESOURCES) if k != dominant]
        n_absent     = int(rng.integers(1, 4))
        absent_set   = set(rng.choice(non_dominant, size=n_absent, replace=False).tolist())

        R0 = np.zeros(N_RESOURCES)
        for k in range(N_RESOURCES):
            if k == dominant:
                R0[k] = rng.uniform(0.5, 2.0) * rng.uniform(3.0, 6.0)
            elif k in absent_set:
                R0[k] = 0.0
            else:
                R0[k] = rng.uniform(0.5, 2.0)

        delta = float(rng.uniform(0.05, 0.20))
        w     = rng.uniform(0.6, 1.2, size=N_RESOURCES)
        l     = rng.uniform(0.1, 0.3,  size=N_RESOURCES)

        broths.append({
            "name"     : BROTH_NAMES[b],
            "delta"    : delta,
            "R0"       : R0,
            "w"        : w,
            "l"        : l,
            "dominant" : dominant,
            "absent"   : sorted(absent_set),
        })
    return broths

In [4]:
# =============================================================================
# 3.  SPARSE CROSS-FEEDING MATRICES
# =============================================================================
def build_species_crossfeed_matrices(species_type):
    D_list = []
    for i in range(N_SPECIES):
        D_i = np.zeros((N_RESOURCES, N_RESOURCES))
        for beta in range(N_RESOURCES):
            n_out        = int(rng.integers(CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX + 1))
            active_alpha = rng.choice(N_RESOURCES, size=n_out, replace=False)

            if species_type[i] == "specialist":
                pref = (i - 1) % N_RESOURCES
                alpha_vec = np.array([
                    rng.uniform(2.0, 5.0) if a == pref and beta != pref
                    else rng.uniform(0.5, 2.0)
                    for a in active_alpha
                ])
            elif species_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=n_out)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=n_out)

            probs = rng.dirichlet(alpha_vec)
            D_i[active_alpha, beta] = probs
        D_list.append(D_i)
    return D_list

In [5]:
# =============================================================================
# 4.  ODE
# =============================================================================
def micrm_ode(t, y, c, w, l, m, g_rate, delta, R0, D_list_sub):
    n_sp  = c.shape[0]
    n_res = c.shape[1]
    N = np.maximum(y[:n_sp],  0.0)
    R = np.maximum(y[n_sp:],  0.0)

    energy_intake = c @ (w * (1.0 - l) * R)
    net_growth    = g_rate * (energy_intake - m)
    dN = N * net_growth - delta * N

    dilution_net = delta * (R0 - R)
    uptake       = (c.T @ N) * R

    eps = 1e-12
    crossfeed = np.zeros(n_res)
    for i in range(n_sp):
        Ni = N[i]
        if Ni < 1e-14:
            continue
        for beta in range(n_res):
            if c[i, beta] < 1e-14:
                continue
            flux_i_beta = l[beta] * c[i, beta] * Ni * R[beta]
            if flux_i_beta < 1e-16:
                continue
            col = D_list_sub[i][:, beta]
            nz  = np.nonzero(col)[0]
            for alpha in nz:
                crossfeed[alpha] += col[alpha] * w[beta] / (w[alpha] + eps) * flux_i_beta

    dR = dilution_net - uptake + crossfeed
    return np.concatenate([dN, dR])


# =============================================================================
# 5.  ODE RUNNER — now with configurable t_end and dense t_eval
# =============================================================================
def run_micrm(c_full, w, l, m_full, g_full, delta, R0, D_list,
              species_subset, t_end=T_MEASURE):
    """
    Run MiCRM ODE for a given species subset.
    t_end: stop time. Default is T_MEASURE (non-steady-state).
           Pass t_end=300.0 to get steady-state (legacy behaviour).

    Returns:
        N_out   : [N_SPECIES] array of final abundances (zeros for absent species)
        R_final : [N_RESOURCES] array of final resource concentrations
        sol     : the full ODE solution object (contains trajectory)
    """
    c_sub      = c_full[species_subset, :]
    m_sub      = m_full[species_subset]
    g_sub      = g_full[species_subset]
    D_list_sub = [D_list[i] for i in species_subset]
    n_sub      = len(species_subset)

    # Initial conditions: all species start at 0.01, resources at R0
    y0 = np.concatenate([np.full(n_sub, 0.01), R0.copy()])

    # t_eval: dense enough to see dynamics, but not wasteful
    n_eval = max(50, int(t_end * 5))   # 5 points per time unit
    sol = solve_ivp(
        fun    = micrm_ode,
        t_span = (0.0, t_end),
        y0     = y0,
        args   = (c_sub, w, l, m_sub, g_sub, delta, R0, D_list_sub),
        method = "RK45",
        t_eval = np.linspace(0, t_end, n_eval),
        rtol   = 1e-5,
        atol   = 1e-7,
    )

    # Extract final state (at t_end, NOT necessarily steady state)
    y_final = sol.y[:, -1]
    N_out = np.zeros(N_SPECIES)
    for local_idx, global_idx in enumerate(species_subset):
        N_out[global_idx] = max(0.0, y_final[local_idx])
    R_final = np.maximum(y_final[n_sub:], 0.0)

    return N_out, R_final, sol

In [6]:
# =============================================================================
# 6.  SUPPRESSION METRICS
# =============================================================================
def calc_suppression(N_pathogen_measured, N_pathogen_solo):
    """
    suppression = 1 - N_path_with_community / N_path_solo
    Clipped to [0, 1]. Negative values (community makes pathogen worse) → 0.
    Note: both values measured at the SAME t_end (T_MEASURE), not steady state.
    """
    eps = 1e-12
    if N_pathogen_solo < eps:
        return 0.0
    raw = 1.0 - (N_pathogen_measured / (N_pathogen_solo + eps))
    return float(np.clip(raw, 0.0, 1.0))

In [10]:
# =============================================================================
# 7.  MAIN DATASET GENERATION
# =============================================================================
def generate_dataset():

    print("=" * 65)
    print("  MiCRM Non-Steady-State Dataset Generator")
    print(f"  T_MEASURE = {T_MEASURE} (non-SS measurement time)")
    print(f"  N_SPECIES = {N_SPECIES} (1 pathogen + {N_COMMUNITY} community strains)")
    print(f"  N_RESOURCES = {N_RESOURCES}, N_GENES = {N_GENES}")
    print(f"  Community sizes: {COMMUNITY_SIZES}")
    print(f"  Samples per size per broth: {N_SAMPLES_PER_SIZE}")
    expected = N_BROTHS * len(COMMUNITY_SIZES) * N_SAMPLES_PER_SIZE
    print(f"  Expected communities: ~{expected}")
    print("=" * 65)

    # ── Build shared ecology ──────────────────────────────────────────────
    print("\n[1] Building species traits ...")
    genes, c_matrix, uptake_mask, species_type, preferred_nutrient = build_species_traits()

    print("[2] Building broth environments ...")
    broths = build_broths()

    print("[3] Building cross-feeding matrices ...")
    D_list = build_species_crossfeed_matrices(species_type)

    # Species-level parameters
    m = rng.uniform(0.1, 0.5, size=N_SPECIES)    # maintenance cost
    g = rng.uniform(0.5, 1.5, size=N_SPECIES)    # growth rate scalar

    # ── STEP A: Pathogen solo at T_MEASURE for each broth ──────────────────
    # IMPORTANT: solo measurement is also at T_MEASURE, not steady state.
    # This ensures ss_luminescence and community_suppression are on the same
    # time scale as the real plate reader experiments.
    print(f"\n[4] Computing pathogen solo abundances at T={T_MEASURE} ...")
    pathogen_solo = {}
    for broth in tqdm(broths, desc="  Pathogen solo", unit="broth"):
        N_s, _, _ = run_micrm(
            c_matrix, broth["w"], broth["l"], m, g,
            broth["delta"], broth["R0"], D_list,
            species_subset=[PATHOGEN_IDX],
            t_end=T_MEASURE
        )
        pathogen_solo[broth["name"]] = float(N_s[PATHOGEN_IDX])
        print(f"    {broth['name']}: N_pathogen_solo(T={T_MEASURE}) = "
              f"{N_s[PATHOGEN_IDX]:.4f}")

    # ── STEP B: Pairwise (single strain + pathogen) at T_MEASURE ───────────
    # This gives ss_luminescence: the individual suppression of each strain.
    # Equivalent to the single-strain plate reader experiment.
    print(f"\n[5] Computing single-strain suppression (ss_lum) at T={T_MEASURE} ...")
    ss_lum = {}  # {broth_name: {species_idx: suppression_score}}
    strain_indices = list(range(1, N_SPECIES))  # all non-pathogen strains

    for broth in tqdm(broths, desc="  Pairwise ODE", unit="broth"):
        bname    = broth["name"]
        N_solo_p = pathogen_solo[bname]
        ss_lum[bname] = {}

        for sp_idx in strain_indices:
            N_pair, _, _ = run_micrm(
                c_matrix, broth["w"], broth["l"], m, g,
                broth["delta"], broth["R0"], D_list,
                species_subset=[sp_idx, PATHOGEN_IDX],
                t_end=T_MEASURE
            )
            score = calc_suppression(N_pair[PATHOGEN_IDX], N_solo_p)
            ss_lum[bname][sp_idx] = score

        scores = list(ss_lum[bname].values())
        print(f"    {bname}: ss_lum range [{min(scores):.3f}, {max(scores):.3f}]")

    # ── STEP C: Community suppression for each (broth, size, sample) ───────
    print(f"\n[6] Computing community suppression at T={T_MEASURE} ...")
    print(f"    Generating {N_SAMPLES_PER_SIZE} communities per size per broth ...")

    community_rows  = []
    community_id_counter = 0

    for broth in tqdm(broths, desc="  Community ODE", unit="broth"):
        bname    = broth["name"]
        N_solo_p = pathogen_solo[bname]

        for comm_size in COMMUNITY_SIZES:
            for sample_i in range(N_SAMPLES_PER_SIZE):

                # Sample comm_size strains WITHOUT replacement from the 100 strains
                members_np = rng.choice(strain_indices, size=comm_size, replace=False)
                members    = sorted(members_np.tolist())

                # Run community ODE at T_MEASURE
                species_subset = members + [PATHOGEN_IDX]
                N_comm, R_comm, _ = run_micrm(
                    c_matrix, broth["w"], broth["l"], m, g,
                    broth["delta"], broth["R0"], D_list,
                    species_subset=species_subset,
                    t_end=T_MEASURE
                )

                comm_supp = calc_suppression(N_comm[PATHOGEN_IDX], N_solo_p)
                comm_id   = f"Comm_{community_id_counter:05d}"
                community_id_counter += 1

                # One row per strain in this community (long format, like paper)
                for sp_idx in members:
                    row = {
                        # ── Community metadata ──────────────────────────────
                        "community_id"           : comm_id,
                        "broth_id"               : bname,
                        "community_size"         : comm_size,
                        "community_suppression"  : round(comm_supp, 6),

                        # ── Broth environment ───────────────────────────────
                        "dominant_nutrient"      : f"c{broth['dominant']+1}",
                        "absent_nutrients"       : str([f"c{k+1}" for k in broth["absent"]]),
                        "delta_per_h"            : round(broth["delta"], 4),

                        # ── Strain identity ─────────────────────────────────
                        "strain_id"              : SPECIES_NAMES[sp_idx],
                        "species_type"           : species_type[sp_idx],
                        "maintenance_cost"       : round(float(m[sp_idx]), 4),
                        "growth_rate"            : round(float(g[sp_idx]), 4),

                        # ── ss_luminescence (individual suppression at T_MEASURE) ─
                        # This is the key node feature for GAT-SS.
                        # Negative because we log-normalise: log10(N_path_pair/N_path_solo)
                        # Values close to 0 = weak suppressor, more negative = stronger
                        "ss_luminescence"        : _lum_transform(
                            ss_lum[bname][sp_idx],
                            pathogen_solo[bname]
                        ),
                        "ss_suppression_raw"     : round(ss_lum[bname][sp_idx], 6),
                    }

                    # ── Genes (8 binary) ────────────────────────────────────
                    for g_idx in range(N_GENES):
                        row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])

                    # ── Uptake rates (20 nutrients) ─────────────────────────
                    for k in range(N_RESOURCES):
                        row[f"uptake_c{k+1}"] = round(float(c_matrix[sp_idx, k]), 6)

                    # ── R0: initial nutrient concentrations (broth) ─────────
                    for k in range(N_RESOURCES):
                        row[f"R0_c{k+1}"] = round(float(broth["R0"][k]), 4)

                    community_rows.append(row)

    df = pd.DataFrame(community_rows)

    print(f"\n  Total rows (strain × community): {len(df)}")
    print(f"  Unique communities: {df['community_id'].nunique()}")
    print(f"\n  community_suppression summary:")
    print(df["community_suppression"].describe().round(4))
    print(f"\n  ss_luminescence summary:")
    print(df["ss_luminescence"].describe().round(4))

    frac_suppressing = (df["community_suppression"] > 0.05).mean()
    frac_full        = (df["community_suppression"] > 0.95).mean()
    print(f"\n  Fraction with suppression > 0.05: {frac_suppressing:.2%}")
    print(f"  Fraction with suppression > 0.95: {frac_full:.2%}")
    print(f"  (Good dataset: want <<100% in 'full' bucket)")

    return df, genes, c_matrix, uptake_mask, m, g, broths, D_list, species_type, ss_lum, pathogen_solo


def _lum_transform(suppression_score, N_path_solo):
    """
    Convert suppression score back to a log-normalised luminescence value
    analogous to the real lab Lum = log10(N_path_pair / N_path_solo).

    suppression_score = 1 - N_path_pair/N_path_solo
    → N_path_pair/N_path_solo = 1 - suppression_score
    → Lum = log10(1 - suppression_score)

    Values range from log10(1) = 0 (no suppression) to
    log10(epsilon) ≈ very negative (complete suppression).
    Clipped to [-10, 0] to match real lab data range.
    """
    eps = 1e-10
    ratio = max(1.0 - suppression_score, eps)
    lum = np.log10(ratio)
    return round(float(np.clip(lum, -10.0, 0.0)), 6)

In [8]:
# =============================================================================
# 8.  MONOCULTURE DATASET (for GAT-GO feature selection)
# =============================================================================
def build_monoculture_dataset(df, genes, c_matrix, m, g, broths,
                               D_list, species_type, ss_lum, pathogen_solo):
    """
    Build a monoculture_dataset.csv equivalent: one row per strain.
    Contains: strain_id, species_type, genes, uptake, R0 (reference broth),
              ss_luminescence (average across broths), and placeholder GO features.

    GO features: we simulate 1000 binary GO orthologues derived from the
    8 binary genes + random co-inheritance structure, matching the paper.
    """
    print("\n[7] Building monoculture dataset (for GAT-GO) ...")

    # Use first broth as reference for R0 values in monoculture file
    ref_broth = broths[0]

    # Average ss_luminescence across all broths for each strain
    mono_rows = []
    for sp_idx in range(1, N_SPECIES):
        lum_vals = [_lum_transform(ss_lum[b["name"]][sp_idx], pathogen_solo[b["name"]])
                    for b in broths]
        avg_lum = float(np.mean(lum_vals))

        row = {
            "strain_id"       : SPECIES_NAMES[sp_idx],
            "species_type"    : species_type[sp_idx],
            "maintenance_cost": round(float(m[sp_idx]), 4),
            "growth_rate"     : round(float(g[sp_idx]), 4),
            "ss_luminescence" : round(avg_lum, 6),
        }

        for g_idx in range(N_GENES):
            row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])

        for k in range(N_RESOURCES):
            row[f"uptake_c{k+1}"]  = round(float(c_matrix[sp_idx, k]), 6)
            row[f"R0_c{k+1}"]      = round(float(ref_broth["R0"][k]), 4)

        # Simulate 1000 GO gene orthologue features.
        # These are derived from the 8 binary genes with co-inheritance noise,
        # giving integer copy numbers (0-3) per gene family.
        # The 8 genes provide the "backbone" signal; 992 are noise/correlated.
        go_vec = _simulate_go_features(genes[sp_idx])
        for go_idx, go_val in enumerate(go_vec):
            row[f"GO_{go_idx+1:05d}"] = int(go_val)

        mono_rows.append(row)

    df_mono = pd.DataFrame(mono_rows)
    print(f"  Monoculture dataset: {len(df_mono)} rows × {len(df_mono.columns)} cols")
    return df_mono


def _simulate_go_features(gene_vec_8, n_go=1000):
    """
    Simulate 1000 GO features from 8 binary genes.
    Each GO feature is a weighted sum of genes + noise → rounded to {0,1,2,3}.
    This mimics how real GO copy numbers reflect genome content.
    """
    go_rng = np.random.default_rng(seed=int(sum(gene_vec_8 * [1,2,4,8,16,32,64,128])) % (2**31))
    W_go   = go_rng.choice([0, 1, 2], size=(8, n_go), p=[0.5, 0.3, 0.2])
    raw    = gene_vec_8.astype(float) @ W_go
    noise  = go_rng.uniform(-0.5, 0.5, size=n_go)
    vals   = np.clip(np.round(raw + noise), 0, 3).astype(int)
    return vals

# =============================================================================
# 9.  COMMUNITY INDEX FILE (one row per community)
# =============================================================================
def build_community_index(df):
    """
    community_index.csv: one row per community_id.
    This is the label file that GAT training uses to look up suppression scores.
    """
    idx = df.groupby("community_id").agg(
        broth_id            = ("broth_id",            "first"),
        community_size      = ("community_size",       "first"),
        community_suppression = ("community_suppression", "first"),
        dominant_nutrient   = ("dominant_nutrient",    "first"),
        n_members           = ("strain_id",            "count"),
    ).reset_index()
    return idx

In [13]:
# =============================================================================
# 10.  FIGURES  — v6  (non-steady-state dataset)
# =============================================================================
#
# WHAT'S HERE vs v5:
# ─────────────────────────────────────────────────────────────────────────────
# KEPT FROM v5 (adapted for new data structure):
#   Fig A  ODE dynamics + nutrient trajectories for one community  (was Fig 1)
#   Fig B  Uptake-rate sparsity heatmap + active-nutrient histogram (was Fig 2)
#   Fig C  Sparse cross-feeding matrices D_i for 8 representative  (was Fig 3)
#   Fig D  ss_lum heatmap (species × broth) + ranked bar chart     (was Fig 5)
#   Fig E  Ecological summary: maintenance, D_i breadth, suppression(was Fig 6)
#   Fig F  ss_lum vs community suppression scatter + broth-level   (was Fig 7)
#
# NEW IN v6 (replacing v5 figs that no longer apply):
#   Fig 1  Dataset quality check — the first thing to inspect after running
#   Fig 2  Suppression gradient with community size
#   Fig 3  ODE dynamics at T_MEASURE for one broth
#   Fig 4  Uptake-rate sparsity + active-nutrient distribution
#   Fig 5  Sparse cross-feeding matrices
#   Fig 6  Community membership by size (new structure: sample × size)
#   Fig 7  ss_lum heatmap (strain × broth) + ranked bar chart
#   Fig 8  Ecological summary (maintenance, D_i, maintenance vs ss_lum)
#   Fig 9  ss_lum vs community suppression — the key model validation plot
#   Fig 10 R0 (nutrient env) heatmap across broths
#   Fig 11 ODE trajectory showing WHEN T_MEASURE is relative to dynamics
# =============================================================================
 
def make_figures(df, df_mono, df_index, genes, c_matrix, uptake_mask,
                 m, g, broths, D_list, species_type, ss_lum, pathogen_solo):
    """
    Generate all diagnostic and MiCRM-characterisation figures.
 
    Parameters
    ----------
    df          : community dataset (one row per strain per community)
    df_mono     : monoculture dataset (one row per strain)
    df_index    : community index (one row per community)
    genes       : [N_SPECIES, N_GENES] binary gene matrix
    c_matrix    : [N_SPECIES, N_RESOURCES] uptake rate matrix
    uptake_mask : [N_SPECIES, N_RESOURCES] bool mask
    m, g        : [N_SPECIES] maintenance and growth rate arrays
    broths      : list of broth dicts
    D_list      : list of [N_RESOURCES, N_RESOURCES] crossfeed matrices
    species_type: list of type strings per species
    ss_lum      : {broth_name: {sp_idx: suppression_score}} raw pairwise scores
    pathogen_solo: {broth_name: float} pathogen abundance at T_MEASURE
    """
    import os
    from scipy.stats import spearmanr
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
 
    print("\n[8] Generating figures ...")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    TEAL   = "#028090"
    GOLD   = "#E8A838"
    NAVY   = "#0F1E2E"
    MUTED  = "#5A7A96"
    RED    = "#E53935"
    GREEN  = "#43A047"
    BLUE   = "#1E88E5"
 
    plt.rcParams.update({
        "figure.facecolor" : "white",
        "axes.facecolor"   : "white",
        "axes.spines.top"  : False,
        "axes.spines.right": False,
        "font.size"        : 10,
    })
 
    # ── Helper: community-level summary DataFrame ─────────────────────────────
    comm_df = (df.groupby("community_id")
                 .agg(size=("community_size","first"),
                      supp=("community_suppression","first"),
                      broth=("broth_id","first"))
                 .reset_index())
 
    # ── Helper: per-strain average ss_lum across broths ──────────────────────
    mean_ss = df_mono["ss_luminescence"].values
    sp_types_comm = df_mono["species_type"].values
 
    # =========================================================================
    # FIG 1 — DATASET QUALITY CHECK (run this first!)
    # =========================================================================
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(
        "Figure 1 — Dataset Quality Check  (non-SS MiCRM, T_MEASURE={:.1f})".format(T_MEASURE),
        fontsize=12, fontweight="bold")
 
    # 1a: Community suppression distribution — should NOT be all 1.0
    axes[0].hist(comm_df["supp"].values, bins=40, color=TEAL, alpha=0.85, edgecolor="white")
    axes[0].axvline(0.95, color=RED, ls="--", lw=1.5, label="0.95 threshold")
    axes[0].set_title("Community Suppression Distribution\n(good: spread across [0,1], not all=1)")
    axes[0].set_xlabel("Community suppression"); axes[0].set_ylabel("Count")
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
    frac_full = (comm_df["supp"] > 0.95).mean()
    axes[0].text(0.02, 0.97, f"{frac_full:.1%} fully suppressed",
                 transform=axes[0].transAxes, va="top", fontsize=9,
                 color=RED if frac_full > 0.5 else GREEN)
 
    # 1b: ss_luminescence distribution — should spread from ~-8 to 0
    axes[1].hist(df_mono["ss_luminescence"].values, bins=40, color=GOLD, alpha=0.85, edgecolor="white")
    axes[1].axvspan(-1.0, 0.0, alpha=0.15, color="green", label="Real lab range [-1, 0]")
    axes[1].set_title("ss_luminescence Distribution\n(overlap with real lab range = good)")
    axes[1].set_xlabel("ss_luminescence (log-normalised)"); axes[1].set_ylabel("Count")
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
 
    # 1c: Pathogen solo abundance per broth — must be > 0 for suppression to be defined
    bnames    = [b["name"] for b in broths]
    solo_vals = [pathogen_solo[b] for b in bnames]
    bars = axes[2].bar(range(len(bnames)), solo_vals, color=BLUE, alpha=0.8, edgecolor="white")
    any_zero = any(v < 1e-4 for v in solo_vals)
    axes[2].set_title(
        f"Pathogen Solo N at T={T_MEASURE}\n"
        f"{'⚠ Some near-zero! Decrease T_MEASURE' if any_zero else '✓ All positive'}",
        color=RED if any_zero else "black")
    axes[2].set_ylabel("N_pathogen_solo"); axes[2].set_xlabel("Broth")
    axes[2].set_xticks(range(len(bnames)))
    axes[2].set_xticklabels([b.replace("Broth_","B") for b in bnames], rotation=45, fontsize=8)
    axes[2].grid(True, alpha=0.3, axis="y")
 
    plt.tight_layout()
    _save(fig, "fig1_dataset_quality_v6.png")
 
    # =========================================================================
    # FIG 2 — SUPPRESSION GRADIENT WITH COMMUNITY SIZE
    # =========================================================================
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Figure 2 — Community Suppression vs Size", fontsize=12, fontweight="bold")
 
    # 2a: box + scatter per size
    for sz in COMMUNITY_SIZES:
        sub = comm_df[comm_df["size"] == sz]["supp"].values
        axes[0].scatter([sz]*len(sub), sub, alpha=0.25, s=12, color=TEAL, zorder=1)
    axes[0].boxplot(
        [comm_df[comm_df["size"]==sz]["supp"].values for sz in COMMUNITY_SIZES],
        positions=COMMUNITY_SIZES, widths=[s*0.35 for s in COMMUNITY_SIZES],
        patch_artist=True,
        boxprops=dict(facecolor="#EAF4F5", color=TEAL),
        medianprops=dict(color=GOLD, lw=2.5),
        whiskerprops=dict(color=TEAL), capprops=dict(color=TEAL),
        flierprops=dict(marker=".", ms=3, alpha=0.3, color=TEAL))
    axes[0].set_title("Suppression Distribution by Size")
    axes[0].set_xlabel("Community size N"); axes[0].set_ylabel("Community suppression")
    axes[0].set_xticks(COMMUNITY_SIZES); axes[0].grid(True, alpha=0.3)
 
    # 2b: mean ± std trend line
    ms  = comm_df.groupby("size")["supp"].mean()
    ss_ = comm_df.groupby("size")["supp"].std()
    axes[1].plot(ms.index, ms.values, "o-", color=TEAL, lw=2.5, ms=7,
                 markerfacecolor="white", markeredgewidth=2)
    axes[1].fill_between(ms.index, ms.values - ss_.values, ms.values + ss_.values,
                         alpha=0.15, color=TEAL)
    axes[1].set_title("Mean ± 1 std Suppression vs Size\n(richer communities → higher suppression?)")
    axes[1].set_xlabel("Community size N"); axes[1].set_ylabel("Community suppression")
    axes[1].set_xticks(COMMUNITY_SIZES); axes[1].grid(True, alpha=0.3)
 
    # 2c: count and fraction fully suppressed per size
    frac_full_by_size = comm_df.groupby("size").apply(lambda x: (x["supp"]>0.95).mean())
    axes[2].bar(frac_full_by_size.index, frac_full_by_size.values,
                color=[RED if v > 0.5 else TEAL for v in frac_full_by_size.values],
                alpha=0.85, edgecolor="white",
                width=[s*0.35 for s in frac_full_by_size.index])
    axes[2].set_title("Fraction Fully Suppressed (>0.95)\nper Size Bin")
    axes[2].set_xlabel("Community size N"); axes[2].set_ylabel("Fraction fully suppressed")
    axes[2].axhline(0.5, color=RED, ls="--", lw=1.2, label="50% threshold")
    axes[2].set_xticks(COMMUNITY_SIZES); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)
 
    plt.tight_layout()
    _save(fig, "fig2_suppression_by_size_v6.png")
 
    # =========================================================================
    # FIG 3 — ODE DYNAMICS AT T_MEASURE (showing WHERE we measure)
    # This is critical: it shows the dynamics aren't at steady state
    # =========================================================================
    # Pick the broth with most interesting dynamics (median suppression)
    target_supp = comm_df["supp"].median()
    best_broth_name = (comm_df
                       .assign(dist=lambda x: (x["supp"] - target_supp).abs())
                       .sort_values("dist")
                       .iloc[0]["broth"])
    broth_obj = next(b for b in broths if b["name"] == best_broth_name)
 
    # Find a community of size 8 from that broth
    comm_ex = (df[(df["broth_id"] == best_broth_name) & (df["community_size"] == 8)]
               .groupby("community_id")["strain_id"].apply(list)
               .reset_index().iloc[0])
    member_ids   = comm_ex["strain_id"]
    member_idx   = [SPECIES_NAMES.index(s) for s in member_ids]
    full_members = member_idx + [PATHOGEN_IDX]
 
    # Run ODE for LONGER than T_MEASURE so we can see the full trajectory
    t_long = T_MEASURE * 6
    _, _, sol_long = run_micrm(c_matrix, broth_obj["w"], broth_obj["l"],
                                m, g, broth_obj["delta"], broth_obj["R0"],
                                D_list, full_members, t_end=t_long)
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"Figure 3 — ODE Dynamics for {best_broth_name} (N=8 community)\n"
        f"Showing T_MEASURE={T_MEASURE} relative to full trajectory",
        fontsize=11, fontweight="bold")
 
    local_pathogen_idx = full_members.index(PATHOGEN_IDX)
    for local_i, global_i in enumerate(full_members):
        col = TYPE_COLORS[species_type[global_i]]
        lw  = 2.5 if global_i == PATHOGEN_IDX else 0.9
        al  = 1.0 if global_i == PATHOGEN_IDX else 0.45
        zo  = 5   if global_i == PATHOGEN_IDX else 1
        axes[0].plot(sol_long.t, sol_long.y[local_i],
                     color=col, lw=lw, alpha=al, zorder=zo)
 
    axes[0].axvline(T_MEASURE, color=GOLD, lw=2.5, ls="--",
                    label=f"T_MEASURE = {T_MEASURE}", zorder=10)
    axes[0].set_yscale("symlog", linthresh=0.001)
    axes[0].set_xlabel("Time"); axes[0].set_ylabel("Abundance N_i (symlog)")
    axes[0].set_title("Species Dynamics\n(vertical line = measurement point)")
    axes[0].legend(handles=[
        Line2D([0],[0], color=TYPE_COLORS["pathogen"],   lw=2.5, label="Pathogen"),
        Line2D([0],[0], color=TYPE_COLORS["specialist"], lw=1.5, label="Specialist"),
        Line2D([0],[0], color=TYPE_COLORS["generalist"], lw=1.5, label="Generalist"),
        Line2D([0],[0], color=GOLD, lw=2.5, ls="--",    label=f"T_MEASURE={T_MEASURE}"),
    ], fontsize=8)
    axes[0].grid(True, alpha=0.3)
 
    # Nutrient dynamics
    n_sub = len(full_members)
    res_colors = plt.cm.tab20(np.linspace(0, 1, N_RESOURCES))
    for k in range(N_RESOURCES):
        lbl = RESOURCE_NAMES[k] + (" (absent)" if k in broth_obj["absent"] else "")
        ls  = "--" if k in broth_obj["absent"] else "-"
        axes[1].plot(sol_long.t, sol_long.y[n_sub + k],
                     color=res_colors[k], lw=1.5, alpha=0.8, ls=ls, label=lbl)
    axes[1].axvline(T_MEASURE, color=GOLD, lw=2.5, ls="--",
                    label=f"T_MEASURE={T_MEASURE}", zorder=10)
    axes[1].set_xlabel("Time"); axes[1].set_ylabel("Nutrient concentration R_α")
    axes[1].set_title("Nutrient Dynamics\n(absent nutrients replenished only by cross-feeding)")
    axes[1].legend(fontsize=5, ncol=2); axes[1].grid(True, alpha=0.3)
 
    plt.tight_layout()
    _save(fig, "fig3_ode_dynamics_v6.png")
 
    # =========================================================================
    # FIG 4 — UPTAKE RATE SPARSITY (MiCRM characteristic)
    # =========================================================================
    sample_idx = ([PATHOGEN_IDX]
                  + list(range(1, min(40, N_SPECIALIST+1)))
                  + list(range(N_SPECIALIST+1, min(N_SPECIALIST+41, N_SPECIES))))
    sample_idx = sample_idx[:80]
 
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("Figure 4 — Uptake Rate Sparsity (MiCRM Trait Structure)",
                 fontsize=12, fontweight="bold")
 
    row_colors = [TYPE_COLORS[species_type[i]] for i in sample_idx]
    sns.heatmap(c_matrix[sample_idx, :], ax=axes[0],
                cmap="YlOrRd", vmin=0, vmax=1,
                xticklabels=RESOURCE_NAMES,
                yticklabels=[SPECIES_NAMES[i].replace("Species_","S") for i in sample_idx],
                linewidths=0.15, linecolor="white",
                cbar_kws={"label": "Uptake rate c_iα"})
    axes[0].set_title("Uptake Rate Matrix c_iα\n(white = masked / unavailable to species)",
                      fontweight="bold")
    for tick, color in zip(axes[0].get_yticklabels(), row_colors):
        tick.set_color(color)
    axes[0].tick_params(axis="y", labelsize=5); axes[0].tick_params(axis="x", labelsize=7)
 
    # Active nutrient count distribution by type
    active_counts = uptake_mask.sum(axis=1)
    for stype, col in TYPE_COLORS.items():
        idx = [i for i, t in enumerate(species_type) if t == stype]
        axes[1].hist([active_counts[i] for i in idx], bins=15,
                     color=col, alpha=0.65,
                     label=f"{stype.capitalize()} (n={len(idx)})", edgecolor="none")
    axes[1].set_xlabel("Number of active nutrients per species")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Uptake Sparsity Distribution\nby Species Type", fontweight="bold")
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
 
    # Gene composition (binary) by type
    gene_cols_labels = [f"g{i+1}" for i in range(N_GENES)]
    for stype, col in TYPE_COLORS.items():
        if stype == "pathogen":
            continue
        idx = [i for i, t in enumerate(species_type) if t == stype]
        gene_means = genes[idx].mean(axis=0)
        axes[2].bar(np.arange(N_GENES) + (0.3 if stype=="specialist" else 0),
                    gene_means, width=0.3, color=col, alpha=0.75,
                    label=stype.capitalize(), edgecolor="white")
    axes[2].set_xlabel("Gene"); axes[2].set_ylabel("Fraction present (=1)")
    axes[2].set_xticks(np.arange(N_GENES) + 0.15)
    axes[2].set_xticklabels(gene_cols_labels)
    axes[2].set_title("Gene Presence Frequency\nby Species Type", fontweight="bold")
    axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3, axis="y")
 
    plt.tight_layout()
    _save(fig, "fig4_uptake_sparsity_v6.png")
 
    # =========================================================================
    # FIG 5 — SPARSE CROSS-FEEDING MATRICES D_i (MiCRM characteristic)
    # =========================================================================
    sample_sp = ([PATHOGEN_IDX, 1, 5, 10,
                  N_SPECIALIST+1, N_SPECIALIST+5,
                  N_SPECIALIST+20, N_SPECIALIST+50])
    sample_sp = [s for s in sample_sp if s < N_SPECIES]
 
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle("Figure 5 — Sparse Cross-Feeding Matrices D_i\n"
                 "(each column sums to ≤ 1; non-zero entries = active cross-feeding routes)",
                 fontsize=11, fontweight="bold")
 
    for ax_idx, sp_idx in enumerate(sample_sp):
        ax = axes.flatten()[ax_idx]
        sns.heatmap(D_list[sp_idx], ax=ax,
                    cmap="YlOrRd", vmin=0, vmax=0.8,
                    xticklabels=[f"in {r}" for r in RESOURCE_NAMES],
                    yticklabels=[f"out {r}" for r in RESOURCE_NAMES],
                    linewidths=0.2, linecolor="lightgray",
                    cbar=(ax_idx == len(sample_sp)-1))
        ax.set_title(f"{SPECIES_NAMES[sp_idx].replace('Species_','S')} "
                     f"[{species_type[sp_idx]}]",
                     fontweight="bold", fontsize=9,
                     color=TYPE_COLORS[species_type[sp_idx]])
        ax.tick_params(axis="both", labelsize=4)
        # Overlay: fraction of non-zero entries
        nz_frac = (D_list[sp_idx] > 0).mean()
        ax.text(0.02, 0.98, f"density={nz_frac:.2f}",
                transform=ax.transAxes, va="top", fontsize=7, color="navy")
 
    plt.tight_layout()
    _save(fig, "fig5_crossfeed_matrices_v6.png")
 
    # =========================================================================
    # FIG 6 — COMMUNITY MEMBERSHIP STRUCTURE (new v6 format)
    # In v6 communities are sampled per size, not per broth → different vis
    # =========================================================================
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Figure 6 — Community Membership Structure\n"
                 "(one community per row, ordered by size then broth)",
                 fontsize=12, fontweight="bold")
 
    # Build membership binary matrix: communities × strains
    # Sample 80 communities for display (10 per size)
    comm_sample = (comm_df.groupby("size").head(10)
                           .reset_index(drop=True))
    comm_ids_show = comm_sample["community_id"].values
    all_strain_names = list(df_mono["strain_id"].values)
    membership_mat = np.zeros((len(comm_ids_show), len(all_strain_names)), dtype=np.int8)
    for row_i, cid in enumerate(comm_ids_show):
        strains_in = df[df["community_id"]==cid]["strain_id"].tolist()
        for s in strains_in:
            if s in all_strain_names:
                membership_mat[row_i, all_strain_names.index(s)] = 1
 
    sns.heatmap(membership_mat, ax=axes[0], cmap="Blues", vmin=0, vmax=1,
                xticklabels=False, yticklabels=False,
                cbar=False)
    axes[0].set_title("Community × Strain Membership\n(blue = present; 80 sampled communities)",
                      fontweight="bold")
    axes[0].set_xlabel("Strain (1–100)"); axes[0].set_ylabel("Community")
 
    # Size distribution
    size_counts = df_index["community_size"].value_counts().sort_index()
    axes[1].bar(size_counts.index, size_counts.values,
                color=TEAL, alpha=0.85, edgecolor="white",
                width=[s*0.35 for s in size_counts.index])
    axes[1].set_xlabel("Community size N"); axes[1].set_ylabel("Number of communities")
    axes[1].set_title("Communities per Size Bin", fontweight="bold")
    axes[1].set_xticks(COMMUNITY_SIZES); axes[1].grid(True, alpha=0.3, axis="y")
 
    # How often is each strain sampled (membership frequency)
    strain_freq = np.zeros(N_SPECIES - 1)
    for _, row_ in df.groupby("community_id")["strain_id"]:
        pass
    # Efficient version: count appearances
    strain_appearance = (df.groupby("strain_id")["community_id"]
                          .nunique()
                          .reindex(all_strain_names, fill_value=0)
                          .values)
    axes[2].hist(strain_appearance, bins=30, color=GOLD, alpha=0.85, edgecolor="white")
    axes[2].set_xlabel("Number of communities strain appears in")
    axes[2].set_ylabel("Count")
    axes[2].set_title("Strain Sampling Frequency\n(should be roughly uniform = random sampling)",
                      fontweight="bold")
    axes[2].axvline(strain_appearance.mean(), color=RED, ls="--", lw=1.5,
                    label=f"Mean={strain_appearance.mean():.1f}")
    axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)
 
    plt.tight_layout()
    _save(fig, "fig6_membership_v6.png")
 
    # =========================================================================
    # FIG 7 — ss_lum HEATMAP + RANKED BAR  (was v5 Fig 5)
    # Rows = strains, columns = broths; cell = pairwise suppression score
    # =========================================================================
    # Build pivot from raw ss_lum dict
    ss_rows = []
    for bname, sp_dict in ss_lum.items():
        for sp_idx, score in sp_dict.items():
            ss_rows.append({"strain_id": SPECIES_NAMES[sp_idx],
                             "broth_id" : bname,
                             "ss_score" : score,
                             "species_type": species_type[sp_idx]})
    ss_df_plot = pd.DataFrame(ss_rows)
 
    pivot_ss = ss_df_plot.pivot_table(index="strain_id", columns="broth_id",
                                       values="ss_score", aggfunc="mean")
    # Sample 60 strains (20 best, 20 worst, 20 median)
    mean_ss_ranked = ss_df_plot.groupby("strain_id")["ss_score"].mean().sort_values()
    n = len(mean_ss_ranked)
    show_ids = (mean_ss_ranked.iloc[:20].index.tolist()
                + mean_ss_ranked.iloc[n//2-10:n//2+10].index.tolist()
                + mean_ss_ranked.iloc[-20:].index.tolist())
    show_ids = list(dict.fromkeys(show_ids))[:60]
    pivot_show = pivot_ss.loc[[s for s in show_ids if s in pivot_ss.index]]
 
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    fig.suptitle("Figure 7 — Pairwise Suppression Heatmap (60-strain sample)\n"
                 "(Green = suppresses pathogen | Red = neutral/slight facilitation)",
                 fontsize=12, fontweight="bold")
 
    sns.heatmap(pivot_show, ax=axes[0], cmap="RdYlGn",
                center=0, vmin=-0.3, vmax=1.0,
                linewidths=0.1, linecolor="lightgray",
                cbar_kws={"label": "Pairwise suppression score"})
    axes[0].set_title("ss_suppression by Strain × Broth\n"
                      "(white = strain not tested in that broth)", fontweight="bold")
    axes[0].tick_params(axis="y", labelsize=6)
 
    mean_pw = (ss_df_plot.groupby(["strain_id","species_type"])["ss_score"]
                         .mean().reset_index()
                         .sort_values("ss_score", ascending=True))
    show_df = pd.concat([mean_pw.head(30), mean_pw.tail(30)]).drop_duplicates()
    axes[1].barh(range(len(show_df)), show_df["ss_score"],
                 color=[TYPE_COLORS[t] for t in show_df["species_type"]],
                 alpha=0.85, edgecolor="white")
    axes[1].set_yticks(range(len(show_df)))
    axes[1].set_yticklabels(
        [s.replace("Species_","S") for s in show_df["strain_id"]], fontsize=7)
    axes[1].axvline(0, color="black", lw=1.5, ls="--")
    axes[1].set_xlabel("Mean pairwise suppression score")
    axes[1].set_title("Ranked Mean Suppression\n(bottom 30 + top 30)", fontweight="bold")
    axes[1].legend(handles=[
        Patch(color=TYPE_COLORS["specialist"], label="Specialist"),
        Patch(color=TYPE_COLORS["generalist"], label="Generalist"),
    ], fontsize=8)
    axes[1].grid(True, alpha=0.3, axis="x")
 
    plt.tight_layout()
    _save(fig, "fig7_pairwise_suppression_v6.png")
 
    # =========================================================================
    # FIG 8 — ECOLOGICAL SUMMARY (MiCRM characteristic traits)
    # =========================================================================
    mean_pw_series = (ss_df_plot.groupby("strain_id")["ss_score"].mean()
                                .reindex(all_strain_names, fill_value=0))
    sp_indices_all = list(range(1, N_SPECIES))
    m_vals   = m[sp_indices_all]
    g_vals   = g[sp_indices_all]
    nz_frac  = np.array([(D_list[i] > 0).mean() for i in sp_indices_all])
    sc_col   = [TYPE_COLORS[species_type[i]] for i in sp_indices_all]
 
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle("Figure 8 — Ecological Summary: MiCRM Trait Relationships",
                 fontsize=12, fontweight="bold")
 
    # 8a: maintenance cost vs mean pairwise suppression
    axes[0,0].scatter(m_vals, mean_pw_series.values,
                      c=sc_col, s=12, alpha=0.5, edgecolors="none")
    rho, pval = spearmanr(m_vals, mean_pw_series.values)
    axes[0,0].set_xlabel("Maintenance cost m_i"); axes[0,0].set_ylabel("Mean pairwise suppression")
    axes[0,0].set_title(f"Maintenance vs Suppression\nSpearman ρ={rho:.3f}, p={pval:.3f}",
                        fontweight="bold")
    axes[0,0].grid(True, alpha=0.3)
 
    # 8b: growth rate vs suppression
    axes[0,1].scatter(g_vals, mean_pw_series.values,
                      c=sc_col, s=12, alpha=0.5, edgecolors="none")
    rho2, _ = spearmanr(g_vals, mean_pw_series.values)
    axes[0,1].set_xlabel("Growth rate scalar g_i"); axes[0,1].set_ylabel("Mean pairwise suppression")
    axes[0,1].set_title(f"Growth Rate vs Suppression\nSpearman ρ={rho2:.3f}", fontweight="bold")
    axes[0,1].grid(True, alpha=0.3)
 
    # 8c: D_i breadth vs suppression
    axes[0,2].scatter(nz_frac, mean_pw_series.values,
                      c=sc_col, s=12, alpha=0.5, edgecolors="none")
    rho3, _ = spearmanr(nz_frac, mean_pw_series.values)
    axes[0,2].set_xlabel("D_i non-zero fraction\n(cross-feeding breadth)")
    axes[0,2].set_ylabel("Mean pairwise suppression")
    axes[0,2].set_title(f"Cross-Feeding Breadth vs Suppression\nSpearman ρ={rho3:.3f}",
                        fontweight="bold")
    axes[0,2].grid(True, alpha=0.3)
    # Type legend
    axes[0,2].legend(handles=[
        Patch(color=TYPE_COLORS["specialist"], label="Specialist"),
        Patch(color=TYPE_COLORS["generalist"], label="Generalist"),
    ], fontsize=8)
 
    # 8d: ss_lum distribution by species type
    for stype, col in TYPE_COLORS.items():
        if stype == "pathogen":
            continue
        vals = ss_df_plot[ss_df_plot["species_type"]==stype]["ss_score"].values
        axes[1,0].hist(vals, bins=30, color=col, alpha=0.6,
                       label=f"{stype.capitalize()} (n={len(vals)})", edgecolor="none")
    axes[1,0].axvline(0, color="black", lw=1.5, ls="--", label="Neutral")
    axes[1,0].set_xlabel("Pairwise suppression score")
    axes[1,0].set_ylabel("Count")
    axes[1,0].set_title("Pairwise Suppression by Species Type", fontweight="bold")
    axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)
 
    # 8e: number of active nutrients vs suppression (sparsity effect)
    active_counts_comm = uptake_mask[sp_indices_all].sum(axis=1)
    axes[1,1].scatter(active_counts_comm, mean_pw_series.values,
                      c=sc_col, s=12, alpha=0.5, edgecolors="none")
    rho4, _ = spearmanr(active_counts_comm, mean_pw_series.values)
    axes[1,1].set_xlabel("Number of active nutrients (uptake breadth)")
    axes[1,1].set_ylabel("Mean pairwise suppression")
    axes[1,1].set_title(f"Uptake Breadth vs Suppression\nSpearman ρ={rho4:.3f}",
                        fontweight="bold")
    axes[1,1].grid(True, alpha=0.3)
 
    # 8f: community suppression per broth (bar chart)
    broth_comm_supp = comm_df.groupby("broth")["supp"].mean().sort_values()
    axes[1,2].barh(range(len(broth_comm_supp)), broth_comm_supp.values,
                   color=TEAL, alpha=0.85, edgecolor="white")
    axes[1,2].set_yticks(range(len(broth_comm_supp)))
    axes[1,2].set_yticklabels(
        [b.replace("Broth_","B") for b in broth_comm_supp.index], fontsize=8)
    axes[1,2].set_xlabel("Mean community suppression")
    axes[1,2].set_title("Mean Community Suppression per Broth\n"
                        "(shows broth-level environmental effect)", fontweight="bold")
    axes[1,2].axvline(comm_df["supp"].mean(), color=GOLD, ls="--", lw=1.5,
                      label=f"Overall mean={comm_df['supp'].mean():.2f}")
    axes[1,2].legend(fontsize=8); axes[1,2].grid(True, alpha=0.3, axis="x")
 
    plt.tight_layout()
    _save(fig, "fig8_ecological_summary_v6.png")
 
    # =========================================================================
    # FIG 9 — ss_lum vs COMMUNITY SUPPRESSION (model validation plot)
    # This is the key figure for GAT: does individual suppression predict community?
    # A WEAK correlation here validates the need for the GNN (community context matters)
    # =========================================================================
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(
        "Figure 9 — ss_lum vs Community Suppression\n"
        "(weak correlation = community context matters = GNN is justified)",
        fontsize=12, fontweight="bold")
 
    # 9a: scatter ss_lum vs community_suppression, coloured by size
    size_cmap = plt.cm.viridis
    size_norm  = plt.Normalize(min(COMMUNITY_SIZES), max(COMMUNITY_SIZES))
    for stype in ["specialist", "generalist"]:
        sub = df[df["species_type"] == stype]
        sc = axes[0].scatter(sub["ss_luminescence"], sub["community_suppression"],
                             c=sub["community_size"].values,
                             cmap=size_cmap, norm=size_norm,
                             s=6, alpha=0.25, edgecolors="none",
                             label=stype.capitalize())
    plt.colorbar(sc, ax=axes[0], label="Community size N")
    rho_sc, pval_sc = spearmanr(df["ss_luminescence"], df["community_suppression"])
    axes[0].set_xlabel("ss_luminescence (individual suppression ability)")
    axes[0].set_ylabel("Community suppression")
    axes[0].set_title(f"Individual vs Community Suppression\n"
                      f"Spearman ρ={rho_sc:.3f} (p={pval_sc:.2e})\n"
                      f"(weak = GNN needed; strong = individual-additive sufficient)",
                      fontweight="bold")
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
 
    # 9b: broth-level mean ss_lum vs mean community suppression
    broth_mean_ss   = df.groupby("broth_id")["ss_luminescence"].mean()
    broth_mean_comm = df.groupby("broth_id")["community_suppression"].mean()
    common_b = broth_mean_ss.index.intersection(broth_mean_comm.index)
    axes[1].scatter(broth_mean_ss[common_b], broth_mean_comm[common_b],
                    s=80, c=TEAL, alpha=0.85, edgecolors="k", zorder=5)
    for bn in common_b:
        axes[1].annotate(bn.replace("Broth_","B"),
                         xy=(broth_mean_ss[bn], broth_mean_comm[bn]),
                         xytext=(3,3), textcoords="offset points", fontsize=7)
    rho_b, _ = spearmanr(broth_mean_ss[common_b], broth_mean_comm[common_b])
    axes[1].set_xlabel("Mean ss_lum per broth")
    axes[1].set_ylabel("Mean community suppression per broth")
    axes[1].set_title(f"Broth-level Correlation\nSpearman ρ={rho_b:.3f}",
                      fontweight="bold")
    axes[1].grid(True, alpha=0.3)
 
    # 9c: mean ss_lum vs community suppression, grouped by community size
    for sz in COMMUNITY_SIZES:
        sub = df[df["community_size"]==sz]
        comm_sub = (sub.groupby("community_id")
                       .agg(mean_ss_lum=("ss_luminescence","mean"),
                            comm_supp=("community_suppression","first"))
                       .reset_index())
        axes[2].scatter(comm_sub["mean_ss_lum"], comm_sub["comm_supp"],
                        s=10, alpha=0.4, edgecolors="none",
                        label=f"N={sz}")
    axes[2].set_xlabel("Mean ss_lum within community")
    axes[2].set_ylabel("Community suppression")
    axes[2].set_title("Mean Member ss_lum vs Community Suppression\nby Size",
                      fontweight="bold")
    axes[2].legend(fontsize=7, ncol=2); axes[2].grid(True, alpha=0.3)
 
    plt.tight_layout()
    _save(fig, "fig9_ss_vs_community_v6.png")
 
    # =========================================================================
    # FIG 10 — NUTRIENT ENVIRONMENT HEATMAP (R0 across broths)
    # Essential for the later nutrient model — shows broth diversity
    # =========================================================================
    R0_matrix = np.array([b["R0"] for b in broths])  # [N_BROTHS, N_RESOURCES]
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Figure 10 — Nutrient Environment (R0) Across Broths\n"
                 "(foundation for the future nutrient prediction model)",
                 fontsize=12, fontweight="bold")
 
    sns.heatmap(R0_matrix, ax=axes[0],
                cmap="YlOrRd", vmin=0,
                xticklabels=RESOURCE_NAMES,
                yticklabels=[b["name"].replace("Broth_","B") for b in broths],
                cbar_kws={"label": "Initial nutrient concentration R0_α"},
                linewidths=0.3, linecolor="white")
    axes[0].set_title("R0 Matrix: Broth × Nutrient\n"
                      "(each row is a distinct chemical environment)",
                      fontweight="bold")
    axes[0].tick_params(axis="x", labelsize=8, rotation=45)
    axes[0].tick_params(axis="y", labelsize=8)
 
    # R0 diversity per broth (how many nutrients are non-zero)
    r0_diversity = (R0_matrix > 0).sum(axis=1)
    r0_richness  = R0_matrix.sum(axis=1)
    color_r0 = plt.cm.viridis(np.linspace(0.2, 0.9, len(broths)))
    for i, b in enumerate(broths):
        axes[1].scatter(r0_diversity[i], r0_richness[i],
                        s=100, color=color_r0[i], edgecolors="k",
                        zorder=5, label=b["name"].replace("Broth_","B"))
        axes[1].annotate(b["name"].replace("Broth_","B"),
                         xy=(r0_diversity[i], r0_richness[i]),
                         xytext=(3,3), textcoords="offset points", fontsize=7)
    axes[1].set_xlabel("Number of non-zero nutrients (nutrient diversity)")
    axes[1].set_ylabel("Total nutrient supply (sum of R0)")
    axes[1].set_title("Broth Nutrient Diversity vs Total Supply\n"
                      "(each dot = one broth environment)",
                      fontweight="bold")
    axes[1].grid(True, alpha=0.3)
 
    plt.tight_layout()
    _save(fig, "fig10_nutrient_environments_v6.png")
 
    # =========================================================================
    # FIG 11 — T_MEASURE SENSITIVITY ANALYSIS
    # Shows what suppression looks like at different measurement times
    # This helps you justify / tune T_MEASURE
    # =========================================================================
    print("    Fig 11: running T_MEASURE sensitivity (may take a few minutes) ...")
 
    # Pick one test community (size 8) from first broth
    test_broth  = broths[0]
    test_comm_ids = df[(df["broth_id"]==test_broth["name"]) &
                       (df["community_size"]==8)]["community_id"].unique()
    if len(test_comm_ids) == 0:
        print("    (skipping Fig 11: no size-8 communities in first broth)")
    else:
        test_cid = test_comm_ids[0]
        test_strains = df[df["community_id"]==test_cid]["strain_id"].tolist()
        test_idx = [SPECIES_NAMES.index(s) for s in test_strains] + [PATHOGEN_IDX]
 
        # Also run pathogen solo across same time points
        t_points = [2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0, 300.0]
        supp_over_time = []
        N_solo_t = {}
        for t_pt in t_points:
            N_s, _, _ = run_micrm(c_matrix, test_broth["w"], test_broth["l"],
                                   m, g, test_broth["delta"], test_broth["R0"],
                                   D_list, [PATHOGEN_IDX], t_end=t_pt)
            N_solo_t[t_pt] = float(N_s[PATHOGEN_IDX])
            N_c, _, _ = run_micrm(c_matrix, test_broth["w"], test_broth["l"],
                                   m, g, test_broth["delta"], test_broth["R0"],
                                   D_list, test_idx, t_end=t_pt)
            supp = calc_suppression(N_c[PATHOGEN_IDX], N_solo_t[t_pt])
            supp_over_time.append(supp)
 
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        fig.suptitle("Figure 11 — T_MEASURE Sensitivity Analysis\n"
                     "(choose T where suppression has good variance, pathogen not zero)",
                     fontsize=11, fontweight="bold")
 
        axes[0].semilogx(t_points, supp_over_time, "o-", color=TEAL, lw=2.5, ms=8,
                         markerfacecolor="white", markeredgewidth=2)
        axes[0].axvline(T_MEASURE, color=GOLD, lw=2.5, ls="--",
                        label=f"Current T_MEASURE={T_MEASURE}")
        axes[0].axhline(0.95, color=RED, lw=1, ls=":", label="0.95 threshold")
        axes[0].set_xlabel("Measurement time T (log scale)")
        axes[0].set_ylabel("Community suppression")
        axes[0].set_title("Suppression vs Measurement Time\n(one test community, N=8)",
                          fontweight="bold")
        axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
 
        axes[1].semilogx(t_points, list(N_solo_t.values()), "s-", color=BLUE, lw=2.5, ms=8,
                         markerfacecolor="white", markeredgewidth=2)
        axes[1].axvline(T_MEASURE, color=GOLD, lw=2.5, ls="--",
                        label=f"T_MEASURE={T_MEASURE}")
        axes[1].set_xlabel("Measurement time T (log scale)")
        axes[1].set_ylabel("N_pathogen_solo")
        axes[1].set_title("Pathogen Solo Abundance vs Time\n"
                          "(should be > 0 at chosen T_MEASURE)",
                          fontweight="bold")
        axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
 
        plt.tight_layout()
        _save(fig, "fig11_tmeasure_sensitivity_v6.png")
 
    print("  All figures saved.")
 
 
# ── Helper: save figure ────────────────────────────────────────────────────
def _save(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    {filename}")
 
 
# =============================================================================
# 11.  MAIN
# =============================================================================
if __name__ == "__main__":
 
    # ── Generate ──────────────────────────────────────────────────────────────
    (df, genes, c_matrix, uptake_mask, m, g, broths, D_list,
     species_type, ss_lum, pathogen_solo) = generate_dataset()
 
    # ── Build auxiliary files ─────────────────────────────────────────────────
    df_mono  = build_monoculture_dataset(df, genes, c_matrix, m, g, broths,
                                          D_list, species_type, ss_lum, pathogen_solo)
    df_index = build_community_index(df)
 
    # ── Save ──────────────────────────────────────────────────────────────────
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    path_comm  = os.path.join(OUTPUT_DIR, "community_dataset_noGO_v6.csv")
    path_mono  = os.path.join(OUTPUT_DIR, "monoculture_dataset_v6.csv")
    path_index = os.path.join(OUTPUT_DIR, "community_index_v6.csv")
 
    df.to_csv(path_comm, index=False)
    df_mono.to_csv(path_mono, index=False)
    df_index.to_csv(path_index, index=False)
 
    print(f"\n{'='*65}")
    print(f"  Saved: {path_comm}")
    print(f"         {len(df)} rows | {df['community_id'].nunique()} communities")
    print(f"  Saved: {path_mono}")
    print(f"         {len(df_mono)} rows (1 per strain)")
    print(f"  Saved: {path_index}")
    print(f"         {len(df_index)} rows (1 per community)")
    print(f"\n  COLUMN GUIDE:")
    print(f"    community_dataset_noGO_v6.csv  → use for GAT-SS")
    print(f"    monoculture_dataset_v6.csv     → use for GAT-GO feature selection")
    print(f"    community_index_v6.csv         → community_suppression labels")
    print(f"\n  KEY CHECKS before running GAT:")
    print(f"    1. community_suppression mean ≠ 1.0 (should be ~0.4-0.7)")
    frac_full = (df_index["community_suppression"] > 0.95).mean()
    print(f"    2. Fraction fully suppressed (>0.95): {frac_full:.1%} (want <30%)")
    print(f"    3. ss_luminescence range: {df['ss_luminescence'].min():.2f} to "
          f"{df['ss_luminescence'].max():.2f} (want spread, not all 0)")
    print(f"\n  If suppression is still all 1.0: decrease T_MEASURE (currently {T_MEASURE})")
    print(f"  If suppression is all 0.0: increase T_MEASURE")
    print(f"{'='*65}")
 
    # ── Figures ───────────────────────────────────────────────────────────────
    make_figures(df, df_mono, df_index, genes, c_matrix, uptake_mask,
                 m, g, broths, D_list, species_type, ss_lum, pathogen_solo)
 
    # ── Final dataset structure summary ───────────────────────────────────────
    print(f"\n  COMMUNITY DATASET COLUMNS ({len(df.columns)} total):")
    for col in df.columns:
        print(f"    {col}")

  MiCRM Non-Steady-State Dataset Generator
  T_MEASURE = 20.0 (non-SS measurement time)
  N_SPECIES = 101 (1 pathogen + 100 community strains)
  N_RESOURCES = 20, N_GENES = 8
  Community sizes: [2, 4, 6, 8, 12, 16, 24, 48]
  Samples per size per broth: 15
  Expected communities: ~1200

[1] Building species traits ...
[2] Building broth environments ...
[3] Building cross-feeding matrices ...

[4] Computing pathogen solo abundances at T=20.0 ...


  Pathogen solo:   0%|                                                                       | 0/10 [00:00<?, ?broth/s]

    Broth_01: N_pathogen_solo(T=20.0) = 2.4862


  Pathogen solo:  30%|██████████████████▉                                            | 3/10 [00:00<00:00, 24.39broth/s]

    Broth_02: N_pathogen_solo(T=20.0) = 2.8845
    Broth_03: N_pathogen_solo(T=20.0) = 2.3594
    Broth_04: N_pathogen_solo(T=20.0) = 4.7246
    Broth_05: N_pathogen_solo(T=20.0) = 2.9598


...

    Broth_09: ss_lum range [0.000, 0.581]


  Pairwise ODE: 100%|███████████████████████████████████████████████████████████████| 10/10 [01:08<00:00,  6.83s/broth]


    Broth_10: ss_lum range [0.000, 0.718]

[6] Computing community suppression at T=20.0 ...
    Generating 15 communities per size per broth ...


  Community ODE: 100%|██████████████████████████████████████████████████████████████| 10/10 [08:03<00:00, 48.37s/broth]



  Total rows (strain × community): 18000
  Unique communities: 1200

  community_suppression summary:
count    18000.0000
mean         0.7539
std          0.2520
min          0.0000
25%          0.6393
50%          0.8475
75%          0.9502
max          0.9987
Name: community_suppression, dtype: float64

  ss_luminescence summary:
count    18000.0000
mean        -0.0523
std          0.0954
min         -1.2919
25%         -0.0708
50%         -0.0085
75%         -0.0000
max         -0.0000
Name: ss_luminescence, dtype: float64

  Fraction with suppression > 0.05: 98.62%
  Fraction with suppression > 0.95: 25.12%
  (Good dataset: want <<100% in 'full' bucket)

[7] Building monoculture dataset (for GAT-GO) ...
  Monoculture dataset: 100 rows × 1053 cols

  Saved: C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\community_dataset_noGO_v6.csv
         18000 rows | 1200 communities
  Saved: C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\monoculture_dataset_v6.csv